# Semaine 3 : Ordonnancement à priorité fixe – RM et DM

## Objectifs pédagogiques
- Comprendre les algorithmes d’ordonnancement à priorité fixe : Rate Monotonic (RM) et Deadline Monotonic (DM).
- Savoir appliquer la condition suffisante de Liu & Layland pour RM.
- Maîtriser l’analyse de temps de réponse (RTA) pour vérifier l’ordonnançabilité exacte.
- Comparer RM et DM lorsque les échéances sont inférieures aux périodes (D < T).
- Simuler un ordonnancement RM pour visualiser le respect des échéances.

## 1. Rappels théoriques

### Ordonnancement à priorité fixe
Chaque tâche se voit attribuer une priorité **fixe** (constante pendant toute la durée de vie du système). L’ordonnanceur exécute toujours la tâche prête de plus haute priorité.

### Rate Monotonic (RM)
- Priorité inversement proportionnelle à la période : plus la période est courte, plus la priorité est élevée.
- **Optimal** parmi tous les algorithmes à priorité fixe **si les échéances sont égales aux périodes (D = T)**.
- Condition suffisante d’ordonnançabilité (Liu & Layland, 1973) :
  $$ U \leq n \left( 2^{1/n} - 1 \right) $$
  où $U = \sum_{i=1}^{n} \frac{C_i}{T_i}$ est l’utilisation totale et $n$ le nombre de tâches.

### Deadline Monotonic (DM)
- Priorité inversement proportionnelle à l’échéance relative : plus l’échéance est courte, plus la priorité est élevée.
- **Optimal** parmi les algorithmes à priorité fixe lorsque $D_i \leq T_i$ pour toutes les tâches.
- Si $D_i = T_i$, DM = RM.
- La condition suffisante de Liu & Layland ne s’applique pas directement si $D_i \neq T_i$ ; on utilise alors l’analyse de temps de réponse (RTA).
Condition suffisante d’ordonnançabilité pour D :
$$ CH \leq n \left( 2^{1/n} - 1 \right) $$
où $CH = \sum_{i=1}^{n} \frac{C_i}{D_i}$ est 
le facteur de charge du processeur et $n$ le nombre de tâches.

## 2. Analyse de temps de réponse (RTA)

Pour une tâche $i$, le **pire temps de réponse** $R_i$ est le temps maximal entre son activation et la fin de son exécution, en considérant toutes les interférences des tâches de plus haute priorité.

**Formule itérative** (pour $D_i \le T_i$) :
$$ R_i^{(0)} = C_i $$
$$ R_i^{(k+1)} = C_i + \sum_{j \in hp(i)} \left\lceil \frac{R_i^{(k)}}{T_j} \right\rceil C_j $$
où $hp(i)$ est l’ensemble des tâches de priorité supérieure à $i$.

La convergence se fait lorsque $R_i^{(k+1)} = R_i^{(k)}$. La tâche est ordonnançable si $R_i \le D_i$.

Si $R_i > D_i$, la tâche risque de rater son échéance. Le système est ordonnançable si **toutes** les tâches vérifient $R_i \le D_i$.

In [ ]:
import math

def liu_layland_condition(taches):
    """
    Vérifie la condition suffisante de Liu & Layland pour RM.
    taches : liste de dictionnaires {'nom': str, 'C': float, 'T': float}
    Retourne (U, borne, resultat_bool)
    """
    n = len(taches)
    U = sum(t['C'] / t['T'] for t in taches)
    borne = n * (2 ** (1 / n) - 1)
    return U, borne, U <= borne

# Exemple : deux tâches
taches_ex = [
    {'nom': 'T1', 'C': 2, 'T': 5},
    {'nom': 'T2', 'C': 3, 'T': 10}
]

U, borne, ok = liu_layland_condition(taches_ex)
print(f"Utilisation U = {U:.4f}, borne = {borne:.4f}")
print("Condition suffisante vérifiée :", ok)

In [ ]:
def analyse_temps_reponse(taches):
    """
    Calcule le pire temps de réponse R pour chaque tâche par RTA.
    taches : liste de dictionnaires {'nom': str, 'C': float, 'T': float, 'D': float}
    Les priorités sont supposées déjà attribuées (RM ou DM).
    Retourne un dictionnaire des R et un booléen indiquant si toutes les tâches respectent leurs échéances.
    """
    # Trier par priorité décroissante (période ou échéance croissante)
    # On suppose que la liste est déjà triée par priorité (la plus haute en premier)
    resultats = {}
    for i, tache in enumerate(taches):
        C_i = tache['C']
        T_i = tache['T']
        D_i = tache.get('D', T_i)  # par défaut D = T
        R = C_i
        while True:
            somme = 0
            for j in range(i):  # tâches de plus haute priorité (indices inférieurs)
                tache_hp = taches[j]
                somme += math.ceil(R / tache_hp['T']) * tache_hp['C']
            nouveau_R = C_i + somme
            if nouveau_R == R:
                break
            R = nouveau_R
            # Sécurité contre les non-convergences
            if R > D_i + 1000:  # on arrête si R explose
                break
        resultats[tache['nom']] = {'R': R, 'D': D_i, 'respect': R <= D_i}
    
    tous_respectes = all(res['respect'] for res in resultats.values())
    return resultats, tous_respectes

# Exemple : trois tâches RM (D = T)
taches_rm = [
    {'nom': 'T1', 'C': 1, 'T': 4, 'D': 4},
    {'nom': 'T2', 'C': 2, 'T': 6, 'D': 6},
    {'nom': 'T3', 'C': 3, 'T': 12, 'D': 12}
]

res, ok = analyse_temps_reponse(taches_rm)
print("Résultats RTA (RM) :")
for nom, info in res.items():
    print(f"  {nom}: R={info['R']}, D={info['D']}, respecte={info['respect']}")
print("Toutes les échéances respectées :", ok)

## 3. Activité 1 : Tester la condition de Liu & Layland

Exécutez la cellule ci-dessous pour calculer l’utilisation et la borne pour différents ensembles de tâches. Modifiez les valeurs de `C` et `T` et observez si la condition suffisante est satisfaite.

**Questions :**
1. Pour 3 tâches, que vaut la borne théorique $n(2^{1/n}-1)$ ?
2. Trouvez un ensemble de tâches dont l’utilisation dépasse la borne mais qui reste ordonnançable (vérifiez avec RTA).
3. Que se passe-t-il si le nombre de tâches augmente ? La borne tend vers quelle valeur ?

In [ ]:
# Modifiez cette liste de tâches (C, T)
taches_test = [
    {'nom': 'T1', 'C': 3, 'T': 10},
    {'nom': 'T2', 'C': 4, 'T': 12},
    {'nom': 'T3', 'C': 5, 'T': 20}
]

U, borne, ok = liu_layland_condition(taches_test)
print(f"Utilisation U = {U:.4f}, borne = {borne:.4f}")
print("Condition suffisante vérifiée :", ok)

# Vérification avec RTA (supposons D = T)
# On doit trier par priorité RM (période croissante)
taches_rm_sorted = sorted(taches_test, key=lambda x: x['T'])
res, ok_rta = analyse_temps_reponse(taches_rm_sorted)
print("\nRésultats RTA :")
for nom, info in res.items():
    print(f"  {nom}: R={info['R']}, D={info['D']}, respecte={info['respect']}")
print("Toutes les échéances respectées (RTA) :", ok_rta)

## 4. Activité 2 : Comparaison RM et DM lorsque D < T

Lorsque les échéances sont inférieures aux périodes ($D_i < T_i$), RM n’est plus optimal. DM donne la priorité selon l’échéance, ce qui peut améliorer l’ordonnançabilité.

Considérons deux tâches :
- T1 : C=2, T=10, D=3
- T2 : C=3, T=12, D=4

**Questions :**
1. Calculez les priorités selon RM et DM.
2. Appliquez l’analyse RTA pour les deux assignations de priorités. Le système est-il ordonnançable avec RM ? Avec DM ?
3. Pourquoi DM est-il meilleur dans ce cas ?

Utilisez le code ci-dessous pour vérifier.

In [ ]:
taches_d_lt_t = [
    {'nom': 'T1', 'C': 2, 'T': 10, 'D': 3},
    {'nom': 'T2', 'C': 3, 'T': 12, 'D': 4}
]

# RM : tri par période croissante
rm_order = sorted(taches_d_lt_t, key=lambda x: x['T'])
print("Ordre RM (période croissante) :", [t['nom'] for t in rm_order])
res_rm, ok_rm = analyse_temps_reponse(rm_order)
print("Résultats RM :")
for nom, info in res_rm.items():
    print(f"  {nom}: R={info['R']}, D={info['D']}, respecte={info['respect']}")
print("RM ordonnançable :", ok_rm)

# DM : tri par échéance croissante
dm_order = sorted(taches_d_lt_t, key=lambda x: x['D'])
print("\nOrdre DM (échéance croissante) :", [t['nom'] for t in dm_order])
res_dm, ok_dm = analyse_temps_reponse(dm_order)
print("Résultats DM :")
for nom, info in res_dm.items():
    print(f"  {nom}: R={info['R']}, D={info['D']}, respecte={info['respect']}")
print("DM ordonnançable :", ok_dm)

## 5. Exercices

### Exercice 1 : Calcul manuel de RTA

Soit le système de trois tâches suivant (RM, D = T) :
- T1 : C=1, T=4
- T2 : C=2, T=6
- T3 : C=3, T=12

1. Vérifiez la condition de Liu & Layland.
2. Calculez manuellement le pire temps de réponse de chaque tâche en utilisant l’itération RTA. Montrez les étapes pour T3.
3. Concluez sur l’ordonnançabilité du système.

### Exercice 2 : Influence de D < T

Considérez deux tâches :
- T1 : C=3, T=10, D=5
- T2 : C=4, T=15, D=8

1. Déterminez l’ordre des priorités avec RM et DM.
2. Calculez R pour chaque tâche dans les deux cas (vous pouvez utiliser le code).
3. Le système est-il ordonnançable avec RM ? Avec DM ?

### Exercice 3 : Limite de la condition suffisante

Montrez par un exemple que la condition de Liu & Layland n’est pas nécessaire : trouvez un ensemble de tâches avec $U > n(2^{1/n}-1)$ mais qui est ordonnançable selon RTA. (Indice : essayez trois tâches avec des périodes proches et des temps d’exécution petits).

## 6. TP : Simulation d’un ordonnanceur RM et vérification RTA

### Objectif
Implémenter une simulation d’ordonnancement RM pour un ensemble de tâches périodiques et comparer les résultats avec l’analyse RTA.

### Étapes
1. Écrivez une fonction `simuler_rm(taches, duree)` qui simule l’exécution de tâches périodiques selon RM (priorité = période la plus courte) sur un processeur. La fonction retourne un chronogramme et la liste des violations d’échéances.
2. Testez avec le système de l’Exercice 1. Vérifiez que la simulation ne montre aucune violation sur une hyperpériode.
3. Modifiez les paramètres pour créer un système non ordonnançable (par exemple en augmentant C). Observez les violations dans la simulation.
4. Pour chaque système testé, exécutez `analyse_temps_reponse` et comparez : la simulation sur une hyperpériode suffit-elle à garantir l’ordonnançabilité ? Pourquoi ?

**Indication :** Vous pouvez réutiliser la fonction de simulation de la semaine 2 en l’adaptant pour prendre en compte plusieurs tâches et l’algorithme RM.

In [ ]:
# Code de simulation RM (à compléter)
def simuler_rm(taches, duree):
    """
    Simule l'ordonnancement RM de tâches périodiques.
    taches : liste de dictionnaires {'nom', 'C', 'T', 'D'}
    Retourne (trace, violations)
    """
    # Initialisation des états
    etat = {}
    for t in taches:
        etat[t['nom']] = {
            'prochaine_activation': 0,
            'prochaine_echeance': t['D'],
            'restant': 0,
            'deadline_violee': False
        }
    
    temps = 0
    trace = []
    violations = []
    
    while temps < duree:
        # Activation des tâches au début de leur période
        for t in taches:
            nom = t['nom']
            if temps >= etat[nom]['prochaine_activation']:
                etat[nom]['restant'] = t['C']
                etat[nom]['prochaine_activation'] += t['T']
                etat[nom]['prochaine_echeance'] = temps + t['D']
                etat[nom]['deadline_violee'] = False
        
        # Choix de la tâche à exécuter : RM (période la plus courte d'abord)
        tache_exec = None
        for t in sorted(taches, key=lambda x: x['T']):
            nom = t['nom']
            if etat[nom]['restant'] > 0:
                tache_exec = t
                break
        
        if tache_exec:
            nom = tache_exec['nom']
            etat[nom]['restant'] -= 1
            trace.append((temps, nom))
            
            # Vérification des échéances pour toutes les tâches actives
            for t in taches:
                nom_t = t['nom']
                if temps >= etat[nom_t]['prochaine_echeance'] and etat[nom_t]['restant'] > 0 and not etat[nom_t]['deadline_violee']:
                    violations.append((temps, nom_t))
                    etat[nom_t]['deadline_violee'] = True
        else:
            trace.append((temps, 'IDLE'))
        
        temps += 1
    
    return trace, violations

# Test avec le système de l'exercice 1
taches_sim = [
    {'nom': 'T1', 'C': 1, 'T': 4, 'D': 4},
    {'nom': 'T2', 'C': 2, 'T': 6, 'D': 6},
    {'nom': 'T3', 'C': 3, 'T': 12, 'D': 12}
]
trace, violations = simuler_rm(taches_sim, 24)  # hyperpériode = 24
print("Chronogramme (extrait) :")
for temps, nom in trace[:24]:
    print(f"{temps:2d} : {nom}")
print("\nViolations :", violations if violations else "aucune")

## Références
- Buttazzo, G. – *Hard Real-Time Computing Systems*, Chapitre 4.
- Liu, J.W.S. – *Real-Time Systems*, Chapitre 6.